# ResNet-18 Transfer Learning — Multi-GPU with Lightning Fabric

Same two-stage strategy as before, now with:
- **Lightning Fabric** — multi-GPU with minimal code changes
- **Rich** progress bars with live metrics
- **TensorBoard** logging
- **ModelCheckpoint** callback
- **EarlyStopping** callback

In [1]:
!pip install lightning torchmetrics rich -q

## 1. Imports and configuration

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchmetrics
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision.models import resnet18, ResNet18_Weights

import lightning as L
from lightning.fabric.loggers import TensorBoardLogger

from rich.progress import (
    Progress, SpinnerColumn, BarColumn,
    TextColumn, TimeElapsedColumn, MofNCompleteColumn,
)
from rich.console import Console
from rich.table import Table

console = Console()

# ── Configuration ─────────────────────────────────────────────────────────────
CFG = dict(
    seed          = 123,
    num_classes   = 10,
    batch_size    = 64,
    num_workers   = 4,
    # Stage 1 — head only
    stage1_epochs = 5,
    stage1_lr     = 1e-3,
    # Stage 2 — full fine-tuning
    stage2_epochs = 45,
    stage2_lr     = 1e-4,
    # Callbacks
    patience      = 10,     # early stopping patience
    save_top_k    = 2,
    save_dir      = "checkpoints/",
    log_dir       = "logs/fabric-two-stage",
    # Fabric
    accelerator   = "auto",  # "cpu" | "gpu" | "auto"
    devices       = "auto",  # 1 | 2 | "auto" (uses all available GPUs)
    strategy      = "auto",  # "ddp" for multi-GPU, "auto" lets Fabric decide
    precision     = "16-mixed",  # mixed precision — comment out if on CPU
)

L.seed_everything(CFG["seed"])
console.print(f"[bold green]Config loaded.[/] Torch {torch.__version__} | Lightning {L.__version__}")

Seed set to 123


Config loaded. Torch 2.8.0+cu128 | Lightning 2.6.0

## 2. Callbacks

Fabric doesn't have built-in callbacks like `LightningModule` does, so we define lightweight ones ourselves. They are plain Python classes — no magic, no inheritance required.

In [2]:
import glob
import os


class ModelCheckpoint:
    """Save the top-k checkpoints ranked by val_acc."""

    def __init__(self, save_dir: str, run_name: str, save_top_k: int = 2):
        self.save_dir   = Path(save_dir)
        self.run_name   = run_name
        self.save_top_k = save_top_k
        self.best_acc   = -1.0
        self.best_path  = None
        self.save_dir.mkdir(parents=True, exist_ok=True)

    def step(self, fabric: L.Fabric, model: nn.Module,
             optimizer: torch.optim.Optimizer,
             epoch: int, val_acc: float) -> bool:
        if val_acc <= self.best_acc:
            return False

        self.best_acc = val_acc
        filename = f"{self.run_name}_epoch{epoch:03d}_acc{val_acc:.4f}.pt"
        path = self.save_dir / filename

        # fabric.save handles distributed saving correctly
        fabric.save(path, {
            "epoch":                epoch,
            "val_acc":              val_acc,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        })
        self.best_path = path

        # Prune old checkpoints, keep top-k
        pattern = str(self.save_dir / f"{self.run_name}_epoch*.pt")
        ckpts = sorted(
            glob.glob(pattern),
            key=lambda p: float(p.rsplit("_acc", 1)[-1].replace(".pt", "")),
            reverse=True,
        )
        for old in ckpts[self.save_top_k:]:
            os.remove(old)

        return True


class EarlyStopping:
    """Stop training when val_acc stops improving."""

    def __init__(self, patience: int = 10, min_delta: float = 1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.best_acc  = -1.0
        self.counter   = 0

    def reset(self):
        """Call between stages to reset the counter."""
        self.best_acc = -1.0
        self.counter  = 0

    @property
    def should_stop(self) -> bool:
        return self.counter >= self.patience

    def step(self, val_acc: float) -> None:
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc = val_acc
            self.counter  = 0
        else:
            self.counter += 1


console.print("[bold green]Callbacks defined.[/]")

Callbacks defined.

## 3. Data

In [3]:
weights            = ResNet18_Weights.IMAGENET1K_V1
preprocess         = weights.transforms()

train_full   = torchvision.datasets.CIFAR10(root="data/", train=True,  download=True, transform=preprocess)
val_full     = torchvision.datasets.CIFAR10(root="data/", train=True,  download=False, transform=preprocess)
test_dataset = torchvision.datasets.CIFAR10(root="data/", train=False, download=True, transform=preprocess)

generator = torch.Generator().manual_seed(CFG["seed"])
indices   = torch.randperm(len(train_full), generator=generator)
n_val     = 5000

train_dataset = Subset(train_full, indices[n_val:])
val_dataset   = Subset(val_full,   indices[:n_val])

loader_kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], pin_memory=True)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)

console.print(f"Train: {len(train_dataset):,}  Val: {len(val_dataset):,}  Test: {len(test_dataset):,}")

Train: 45,000  Val: 5,000  Test: 10,000

## 4. Model

In [4]:
model = resnet18(weights=weights)

# Freeze all backbone layers
for param in model.parameters():
    param.requires_grad = False

# Replace head — new layer has requires_grad=True by default
model.fc = nn.Linear(model.fc.in_features, CFG["num_classes"])

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
console.print(f"Parameters — total: {total:,}  trainable: {trainable:,}  frozen: {total-trainable:,}")

Parameters — total: 11,181,642  trainable: 5,130  frozen: 11,176,512

## 5. Training loops

The loops are identical to the original notebook. The only Fabric-specific change is using `fabric.backward(loss)` instead of `loss.backward()` — this handles gradient scaling for mixed precision automatically.

In [6]:
def train_one_epoch(fabric, model, loader, optimizer, criterion, acc_metric):
    model.train()
    acc_metric.reset()
    total_loss = 0.0

    for images, labels in loader:
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)

        fabric.backward(loss)   # handles mixed precision scaling
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()


@torch.inference_mode()
def evaluate(model, loader, criterion, acc_metric):
    model.eval()
    acc_metric.reset()
    total_loss = 0.0

    for images, labels in loader:
        logits      = model(images)
        total_loss += criterion(logits, labels).item() * labels.size(0)
        acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()


console.print("[bold green]Loops defined.[/]")

Loops defined.

## 6. Fabric setup

`Fabric` is the single object that handles:
- device placement (`accelerator`, `devices`)
- distributed strategy (`ddp`, `fsdp`, etc.)
- mixed precision (`precision`)
- logging

Calling `fabric.setup()` wraps the model and optimizer. Calling `fabric.setup_dataloaders()` adds a `DistributedSampler` automatically when using multiple GPUs.

In [7]:
logger = TensorBoardLogger(root_dir=CFG["log_dir"], name="resnet18")

fabric = L.Fabric(
    accelerator = CFG["accelerator"],
    devices     = CFG["devices"],
    strategy    = CFG["strategy"],
    precision   = CFG["precision"],
    loggers     = [logger],
)
fabric.launch()

# Move model and optimizer onto the right device(s)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["stage1_lr"],
)

model, optimizer         = fabric.setup(model, optimizer)
train_loader, val_loader = fabric.setup_dataloaders(train_loader, val_loader)

# Metrics live on the same device as the model
device     = fabric.device
train_acc  = torchmetrics.Accuracy(task="multiclass", num_classes=CFG["num_classes"]).to(device)
val_acc    = torchmetrics.Accuracy(task="multiclass", num_classes=CFG["num_classes"]).to(device)

# Callbacks
ckpt_cb    = ModelCheckpoint(CFG["save_dir"], "resnet18-fabric", CFG["save_top_k"])
early_cb   = EarlyStopping(patience=CFG["patience"])

console.print(f"[bold green]Fabric ready.[/] Device: {device}  Devices: {fabric.world_size}")

Using 16-bit Automatic Mixed Precision (AMP)
You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Fabric ready. Device: cuda:0  Devices: 1

## 7. Stage 1 — head only

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

progress = Progress(
    SpinnerColumn(),
    TextColumn("[bold]{task.description}"),
    BarColumn(),
    MofNCompleteColumn(),
    TextColumn("loss={task.fields[loss]:.4f} acc={task.fields[acc]:.4f}"),
    TimeElapsedColumn(),
)

console.rule("[bold cyan]Stage 1 — FC head only")

with progress:
    task = progress.add_task(
        "Stage 1",
        total=CFG["stage1_epochs"],
        loss=0.0, acc=0.0,
    )
    for epoch in range(1, CFG["stage1_epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(fabric, model, train_loader, optimizer, criterion, train_acc)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion, val_acc)

        # Log to TensorBoard via Fabric
        fabric.log_dict({
            "Loss/train": tr_loss, "Loss/val": vl_loss,
            "Acc/train":  tr_acc,  "Acc/val":  vl_acc,
            "Stage":      1,
        }, step=epoch)

        saved = ckpt_cb.step(fabric, model, optimizer, epoch, vl_acc)
        early_cb.step(vl_acc)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        progress.update(task, advance=1, loss=vl_loss, acc=vl_acc)
        console.print(
            f"  Epoch {epoch:02d}/{CFG['stage1_epochs']} | "
            f"train loss {tr_loss:.4f}  acc {tr_acc:.4f} | "
            f"val loss {vl_loss:.4f}  acc {vl_acc:.4f}"
            + (" [green]✓ saved[/]" if saved else "")
        )

        if early_cb.should_stop:
            console.print("[yellow]Early stopping triggered.[/]")
            break

console.print(f"Stage 1 done. Best val acc: {ckpt_cb.best_acc:.4f}")

───────────────────────────────────────────── Stage 1 — FC head only ──────────────────────────────────────────────

Output()

Epoch 01/5 | train loss 0.9321  acc 0.6970 | val loss 0.7413  acc 0.7458 ✓ saved

## 8. Stage 2 — full fine-tuning

In [ ]:
# Unfreeze all backbone parameters
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
console.print(f"All {trainable:,} parameters now trainable.")

# New optimizer — lower LR, fresh momentum state
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["stage2_lr"])
model, optimizer = fabric.setup(model, optimizer)

# Reset early stopping patience for stage 2
early_cb.reset()

s1_epochs  = len(history["train_loss"])   # actual epochs run (may be < stage1_epochs if early stop)
total_epochs = s1_epochs + CFG["stage2_epochs"]

console.rule("[bold cyan]Stage 2 — full fine-tuning")

with progress:
    task = progress.add_task(
        "Stage 2",
        total=CFG["stage2_epochs"],
        loss=0.0, acc=0.0,
    )
    for epoch in range(s1_epochs + 1, total_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(fabric, model, train_loader, optimizer, criterion, train_acc)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion, val_acc)

        fabric.log_dict({
            "Loss/train": tr_loss, "Loss/val": vl_loss,
            "Acc/train":  tr_acc,  "Acc/val":  vl_acc,
            "Stage":      2,
        }, step=epoch)

        saved = ckpt_cb.step(fabric, model, optimizer, epoch, vl_acc)
        early_cb.step(vl_acc)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        progress.update(task, advance=1, loss=vl_loss, acc=vl_acc)
        console.print(
            f"  Epoch {epoch:02d}/{total_epochs} | "
            f"train loss {tr_loss:.4f}  acc {tr_acc:.4f} | "
            f"val loss {vl_loss:.4f}  acc {vl_acc:.4f}"
            + (" [green]✓ saved[/]" if saved else "")
        )

        if early_cb.should_stop:
            console.print("[yellow]Early stopping triggered.[/]")
            break

console.print(f"Stage 2 done. Best val acc: {ckpt_cb.best_acc:.4f}")
console.print(f"Best checkpoint: {ckpt_cb.best_path}")

## 9. Training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
stage_boundary = CFG["stage1_epochs"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["val_loss"],   label="val")
ax1.axvline(stage_boundary, color="gray", linestyle="--", linewidth=1, label="stage 1→2")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()
ax1.spines[["top", "right"]].set_visible(False)

ax2.plot(epochs, history["train_acc"], label="train")
ax2.plot(epochs, history["val_acc"],   label="val")
ax2.axvline(stage_boundary, color="gray", linestyle="--", linewidth=1, label="stage 1→2")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend()
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## 10. Test set evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

CLASSES = ('airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck')

model.eval()
all_preds, all_labels = [], []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device)
        preds  = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=axes[0], colorbar=False, cmap="Blues", xticks_rotation=45)
axes[0].set_title("Confusion matrix")
ConfusionMatrixDisplay(cm.astype(float) / cm.sum(axis=1, keepdims=True),
                       display_labels=CLASSES).plot(
    ax=axes[1], colorbar=False, cmap="Blues",
    values_format=".2f", xticks_rotation=45)
axes[1].set_title("Normalized")
plt.tight_layout()
plt.show()

## 11. TensorBoard

Launch inside the notebook or in a terminal.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/